# Matching strategies: de-novo states → ENCODE reference

Strict one-to-one Hungarian objectives for mapping de-novo ChromHMM states to the published ENCODE 15-state reference, with $J$ the Jaccard index and $E$ the emission similarity of a (work, reference) state pair:

1. **Overlap only**: $\max \sum_i |W_i \cap R_{\pi(i)}|$
2. **Emissions only**: $\max \sum_i E(w_i, \pi(w_i))$
3. **Jaccard + emissions**: $\max \sum_i \alpha J + \beta E$
4. **Concave Jaccard + emissions**: $\max \sum_i \alpha \sqrt{J} + \beta E$ — what `match.match_states` optimises, at the weights set in match.py (`JACCARD_WEIGHT`, `EMISSION_WEIGHT`). Its own mapping is reported as **used (match.py)**.
5. **NPMI + emissions**: $\max \sum_i \alpha\,\mathrm{NPMI} + \beta E$

over $(\alpha, \beta) \in \{(1, 0), (0.8, 0.2), (0.5, 0.5), (0.2, 0.8)\}$.

Cohen's $\kappa$ is not among them: for states much smaller than the genome $\kappa \approx$ Dice $= 2J/(1+J)$, monotone in $J$, so it reproduces the Jaccard mappings.

Datasets in the config `outliers` list are excluded.

In [20]:
import io
import os
import sys
from collections import defaultdict
from contextlib import redirect_stderr
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from scipy.optimize import linear_sum_assignment
from scipy.stats import spearmanr, wilcoxon

for path in ["scripts/rules", "scripts/analysis"]:
    full_path = os.path.abspath(path)
    if full_path not in sys.path:
        sys.path.insert(0, full_path)

import analyze
import match
import summary_plots
import utils
from display import show_table

# The project's plot style (fonts, PDF output) from utils, set once for every plot below.
utils.set_style()

with open("config_encode.yaml") as f:
    config = yaml.safe_load(f)

workdir = os.path.expanduser(config.get("workdir", "~/data/2026_segmentations/encode"))
OUTLIERS = set(config.get("outliers", []))
print(f"Working directory: {workdir}")
print(f"Excluded outliers: {sorted(OUTLIERS)}")

OUT = os.path.join(workdir, "match")
os.makedirs(OUT, exist_ok=True)

Working directory: /Users/Oleg.Shpynov/data/2026_segmentations/encode
Excluded outliers: ['neurosphere', 'thyroid_gland', 'uterus']


## 1. Evaluation

Per dataset and strategy:
- **BP accuracy**: genome share where the mapped work state equals the reference state. In-sample — the overlap objective maximises it by construction, so it bounds the others rather than ranking them.
- **Balanced accuracy**: unweighted mean recall over reference states, $\frac{1}{K} \sum_k |W_{\pi^{-1}(k)} \cap R_k| / |R_k|$.
- **Regulatory / background accuracy**: bp accuracy over reference states typed by `utils` — regulatory = promoter, enhancer and bivalent states; background = `is_noqh` (Quies, Het, ZNF/Rpts, ReprPC).
- **Emission similarity**: mean $E$ over matched pairs.
- **Functional F1, recall, precision**, against annotations not used in matching: active open states vs ATAC-seq, core Tss states vs expressed and RefSeq TSS, Tx states vs expressed gene bodies. Annotation intervals are merged first, as in `analyze.py`. Each benchmark is averaged only over the datasets that have it; `mean_f1` is their mean, defined only for datasets that have all four.

In [2]:
WEIGHTS = [(1.0, 0.0), (0.8, 0.2), (0.5, 0.5), (0.2, 0.8)]
FAMILIES = ["jaccard", "convex_jaccard", "npmi"]
FUNCTIONAL = {
    # benchmark: (state family of the mapped reference label, annotation)
    "active_atac": utils.is_active_open,
    "tss_exptss": utils.is_promoter_core,
    "tss_refseq": utils.is_promoter_core,
    "tx": utils.is_tx,
}


def is_regulatory(name):
    return utils.is_promoter(name) or utils.is_enhancer(name) or utils.is_biv(name)


def merged(segs):
    # One unnamed interval per covered stretch, so overlapping features are not counted twice.
    by_chrom = {}
    for row in segs:
        by_chrom.setdefault(row[0], []).append([row[1], row[2]])
    return [(chrom, s, e, "feature") for chrom in sorted(by_chrom)
            for s, e in analyze.merge_intervals(by_chrom[chrom])]


def npmi_matrix(overlap, wl, rl, w_states, r_states, genome_len):
    mat = np.full((len(w_states), len(r_states)), -1.0)
    for i, w in enumerate(w_states):
        pw = wl[w] / genome_len
        for j, r in enumerate(r_states):
            pr, pwr = rl[r] / genome_len, overlap.get((w, r), 0) / genome_len
            if 0 < pwr < 1:
                mat[i, j] = np.log(pwr / (pw * pr)) / -np.log(pwr)
    return mat


def hungarian(mat, w_states, r_states):
    rows, cols = linear_sum_assignment(mat, maximize=True)
    return {w_states[i]: r_states[j] for i, j in zip(rows, cols)}


def work_segmentation_path(workdir, ds, cell):
    # The de-novo 15-state ChromHMM segmentation of a dataset, the dense BED if there is one.
    default_dir = os.path.join(workdir, ds, "chromhmm_default_result")
    path = os.path.join(default_dir, f"{cell}_15_dense.bed")
    return path if os.path.exists(path) else os.path.join(default_dir, f"{cell}_15_segments.bed")


def evaluate_matching_strategies(config, workdir):
    # Run every matching strategy on every non-outlier ENCODE dataset.
    gtf_path = os.path.join(workdir, "gencode.v46.basic.annotation.gtf.gz")
    refseq_tss_segs = merged(match.load_bed(
        os.path.join(workdir, "ChromHMM", "COORDS", "hg38", "RefSeqTSS2kb.hg38.bed.gz")))

    rows, matching_states_records = [], []
    for ds, ds_cfg in config["datasets"].items():
        if ds in OUTLIERS:
            continue
        cell = ds_cfg["cell"]
        ref_acc = ds_cfg.get("ref_chromhmm")
        if not ref_acc:
            continue

        ref_segs_path = os.path.join(workdir, ds, f"{ref_acc}_chromhmm.bed")
        if not os.path.exists(ref_segs_path):
            ref_segs_path += ".gz"
        default_dir = os.path.join(workdir, ds, "chromhmm_default_result")
        work_segs_path = work_segmentation_path(workdir, ds, cell)
        if not os.path.exists(ref_segs_path) or not os.path.exists(work_segs_path):
            continue

        work_segs = match.load_bed(work_segs_path)
        ref_segs = match.load_bed(ref_segs_path)
        wl = match.state_lengths(work_segs)
        rl = match.state_lengths(ref_segs)
        overlap = match.pair_overlap(ref_segs, work_segs)
        w_states = sorted(wl, key=match._natural_sort_key)
        r_states = sorted(rl, key=match._natural_sort_key)
        w_idx = {s: i for i, s in enumerate(w_states)}
        r_idx = {s: i for i, s in enumerate(r_states)}
        genome_len = sum(rl.values()) or 1

        regulatory = [r for r in r_states if is_regulatory(r)]
        background = [r for r in r_states if utils.is_noqh(r)]
        regulatory_bp = sum(rl[r] for r in regulatory)
        background_bp = sum(rl[r] for r in background)

        # Functional annotations: {benchmark: (per-work-state overlap, annotation bp)}
        annotations = {"tss_refseq": refseq_tss_segs}
        atac_acc = ds_cfg.get("atac")
        atac_path = os.path.join(workdir, ds, f"atac_{atac_acc}.bed.gz") if atac_acc else None
        if atac_path and os.path.exists(atac_path):
            annotations["active_atac"] = merged(match.load_bed(atac_path))
        rnaseq_acc = ds_cfg.get("rnaseq")
        rnaseq_path = os.path.join(workdir, ds, f"rnaseq_{rnaseq_acc}.tsv") if rnaseq_acc else None
        if rnaseq_path and os.path.exists(rnaseq_path):
            exp_ann = dict(analyze.make_expressed_annotations(rnaseq_path, gtf_path))
            if exp_ann.get("ExpressedTSS2kb"):
                annotations["tss_exptss"] = merged(exp_ann["ExpressedTSS2kb"])
            if exp_ann.get("ExpressedGeneBodies"):
                annotations["tx"] = merged(exp_ann["ExpressedGeneBodies"])
        features = {}
        for name, segs in annotations.items():
            ov = {w: v for (w, _), v in match.pair_overlap(segs, work_segs).items()}
            features[name] = (ov, sum(e - s for _, s, e, _ in segs))

        work_em = os.path.join(default_dir, f"{cell}_15_dense.bin_emissions.npz")
        if not os.path.exists(work_em):
            work_em = os.path.join(default_dir, f"{cell}_15_dense.bw_emissions.npz")
        ref_em = os.path.join(workdir, ds, f"{ref_acc}_chromhmm.bw_emissions.npz")
        if not os.path.exists(ref_em):
            ref_em = os.path.join(workdir, ds, f"{ref_acc}_chromhmm.bin_emissions.npz")

        # The used mapping, from the project's own matcher; it also writes the matrices to OUT.
        used = match.match_states([ref_segs], [work_segs], matrix_out=os.path.join(OUT, ds),
                                       work_emissions_path=work_em, ref_emissions_path=ref_em)
        # match_states already reported the emission inputs, silence the second load.
        with redirect_stderr(io.StringIO()):
            em_sim = match._load_emission_similarity(work_em, ref_em, w_states, r_states)
        jacc = match.jaccard_matrix(overlap, wl, rl, w_states, r_states)
        utility = (match.JACCARD_WEIGHT * np.sqrt(jacc) + match.EMISSION_WEIGHT * em_sim
                   if em_sim is not None else np.sqrt(jacc))

        for w in w_states:
            r = used.get(w, w)
            i, j = w_idx[w], r_idx.get(r)
            matching_states_records.append({
                "dataset": ds, "cell": cell, "work_state": w, "matched_ref_state": r,
                "jaccard": jacc[i, j] if j is not None else np.nan,
                "emission_sim": em_sim[i, j] if em_sim is not None and j is not None else np.nan,
                "utility": utility[i, j] if j is not None else np.nan,
            })

        base = {
            "jaccard": jacc,
            "convex_jaccard": np.sqrt(jacc),
            "npmi": npmi_matrix(overlap, wl, rl, w_states, r_states, genome_len),
        }
        ov_mat = np.array([[overlap.get((w, r), 0) for r in r_states] for w in w_states], dtype=float)
        mappings = {"overlap": hungarian(ov_mat, w_states, r_states)}
        for family in FAMILIES:
            for a, b in WEIGHTS:
                if b > 0 and em_sim is None:
                    continue
                mat = a * base[family] + (b * em_sim if b > 0 else 0.0)
                mappings[f"{family}_em_{a}_{b}"] = hungarian(mat, w_states, r_states)
        if em_sim is not None:
            mappings["emissions"] = hungarian(em_sim, w_states, r_states)
        mappings["used"] = {w: r for w, r in used.items() if w in w_idx and r in r_idx}

        for s_name, mapping in mappings.items():
            inv_map = {r: w for w, r in mapping.items()}
            hit = {r: overlap.get((inv_map[r], r), 0) if r in inv_map else 0 for r in r_states}
            row = {
                "dataset": ds,
                "strategy": s_name,
                "mapping": ";".join(f"{w}={mapping.get(w, '')}" for w in w_states),
                "accuracy": sum(hit.values()) / genome_len,
                "balanced_accuracy": float(np.mean([hit[r] / rl[r] for r in r_states if rl[r] > 0])),
                "accuracy_regulatory": sum(hit[r] for r in regulatory) / regulatory_bp if regulatory_bp else np.nan,
                "accuracy_background": sum(hit[r] for r in background) / background_bp if background_bp else np.nan,
                "mean_matched_emission_cos": (float(np.mean([em_sim[w_idx[w], r_idx[r]] for w, r in mapping.items()]))
                                              if em_sim is not None else np.nan),
            }
            for name, is_family in FUNCTIONAL.items():
                f1 = prec = rec = np.nan
                if name in features:
                    ov, feature_bp = features[name]
                    family = [w for w, r in mapping.items() if is_family(r)]
                    state_bp = sum(wl[w] for w in family)
                    hit_bp = sum(ov.get(w, 0) for w in family)
                    f1 = 2.0 * hit_bp / (state_bp + feature_bp) if state_bp + feature_bp else 0.0
                    prec = hit_bp / state_bp if state_bp else 0.0
                    rec = hit_bp / feature_bp if feature_bp else 0.0
                row.update({f"f1_{name}": f1, f"precision_{name}": prec, f"recall_{name}": rec})
            # NaN unless all four benchmarks exist, so the mean is one quantity across datasets.
            row["mean_f1"] = float(np.mean([row[f"f1_{name}"] for name in FUNCTIONAL]))
            rows.append(row)
        print(f"Finished {ds}: {len(mappings)} strategies, functional benchmarks {sorted(features)}")

    pd.DataFrame(matching_states_records).to_csv(os.path.join(OUT, "matching_states.tsv"), sep="\t", index=False)
    return pd.DataFrame(rows)

In [3]:
df_results = evaluate_matching_strategies(config, workdir)
df_results.to_csv(f"{OUT}/matching_strategies_metrics.tsv", sep="\t", index=False)

WEIGHT_TAGS = [f"{a}_{b}" for a, b in WEIGHTS]
FAMILY_LABELS = {"jaccard": "Jaccard", "convex_jaccard": "√Jaccard", "npmi": "NPMI"}
FAMILY_STYLE = {
    # family: (marker, colours from alpha = 1.0 to 0.2)
    "jaccard": ("s", ["#1E8449", "#27AE60", "#52BE80", "#A9DFBF"]),
    "convex_jaccard": ("D", ["#BA4A00", "#E8833A", "#F39C12", "#F8C471"]),
    "npmi": ("v", ["#922B21", "#C0392B", "#E74C3C", "#F1948A"]),
}

strategy_order = ["overlap"]
strategy_labels = {"overlap": "Overlap only"}
strategy_colors = {"overlap": "#4878CF"}
strategy_markers = {"overlap": "X"}
linear_combination_groups = []
for family, (marker, colors) in FAMILY_STYLE.items():
    group = [f"{family}_em_{tag}" for tag in WEIGHT_TAGS]
    for s, (a, b), color in zip(group, WEIGHTS, colors):
        strategy_order.append(s)
        strategy_labels[s] = f"{FAMILY_LABELS[family]} only" if b == 0 else f"{FAMILY_LABELS[family]} + Em ({a}, {b})"
        strategy_colors[s] = color
        strategy_markers[s] = marker
    linear_combination_groups.append((group, colors[1]))
strategy_order += ["emissions", "used"]
strategy_labels.update({"emissions": "Emissions only", "used": "Used"})
strategy_colors.update({"emissions": "#404040", "used": "#000000"})
strategy_markers.update({"emissions": "o", "used": "*"})
strategy_order = [s for s in strategy_order if s in set(df_results["strategy"])]

# The used mapping should be the √Jaccard strategy at match.py's weights.
used_twin = f"convex_jaccard_em_{match.JACCARD_WEIGHT}_{match.EMISSION_WEIGHT}"
maps = df_results.pivot(index="dataset", columns="strategy", values="mapping")
if used_twin in maps:
    print(f"used == {used_twin} on {(maps['used'] == maps[used_twin]).sum()}/{len(maps)} datasets")

SUMMARY_METRICS = ["accuracy", "balanced_accuracy", "accuracy_regulatory", "accuracy_background",
                   "mean_matched_emission_cos"] + [f"f1_{name}" for name in FUNCTIONAL] + ["mean_f1"]
summary = df_results.groupby("strategy")[SUMMARY_METRICS].mean().loc[strategy_order]
summary.insert(0, "label", [strategy_labels[s] for s in summary.index])
summary.round(4).to_csv(f"{OUT}/matching_strategies_summary.tsv", sep="\t")
n_datasets = df_results[df_results["strategy"] == "overlap"][SUMMARY_METRICS].count()
print(f"Evaluated {df_results['dataset'].nunique()} datasets across {len(strategy_order)} strategies.")
print("Datasets per metric:", n_datasets.to_dict())

  RNA-seq: top 12000 expressed genes selected
  GTF: 12000 exp bodies, 46619 non-exp bodies
Incorporating emission similarity (marks: ['H3K27AC', 'H3K27ME3', 'H3K36ME3', 'H3K4ME1', 'H3K4ME3', 'H3K9ME3']; work binarized, reference bigwig)
avg_similarity = 0.5098 (jaccard) 0.9017 (emission) 0.7057 (quality) over 15/15 matched states, bp accuracy 0.6600
  1 -> Quies  (jaccard=0.6366, emission=1.0000, quality=0.8183)
  2 -> Het  (jaccard=0.8285, emission=0.9773, quality=0.9029)
  3 -> ZNF/Rpts  (jaccard=0.5622, emission=0.9644, quality=0.7633)
  4 -> Enh1  (jaccard=0.0030, emission=0.9655, quality=0.4843)
  5 -> Tx  (jaccard=0.7205, emission=0.9976, quality=0.8591)
  6 -> EnhG2  (jaccard=0.3046, emission=0.8636, quality=0.5841)
  7 -> TxWk  (jaccard=0.5768, emission=0.9008, quality=0.7388)
  8 -> Enh2  (jaccard=0.6891, emission=0.9647, quality=0.8269)
  9 -> TssFlnkU  (jaccard=0.7187, emission=0.9837, quality=0.8512)
  10 -> Tss  (jaccard=0.8945, emission=0.9997, quality=0.9471)
  11 -> En

Finished imr90: 15 strategies, functional benchmarks ['active_atac', 'tss_exptss', 'tss_refseq', 'tx']


  RNA-seq: top 12000 expressed genes selected
  GTF: 12000 exp bodies, 44566 non-exp bodies
Incorporating emission similarity (marks: ['H3K27AC', 'H3K27ME3', 'H3K36ME3', 'H3K4ME1', 'H3K4ME3', 'H3K9ME3']; work binarized, reference bigwig)
avg_similarity = 0.5619 (jaccard) 0.9159 (emission) 0.7389 (quality) over 15/15 matched states, bp accuracy 0.8619
  1 -> Tss  (jaccard=0.8808, emission=0.9994, quality=0.9401)
  2 -> TssFlnkU  (jaccard=0.8688, emission=0.9830, quality=0.9259)
  3 -> TssFlnkD  (jaccard=0.6571, emission=0.9100, quality=0.7835)
  4 -> Biv  (jaccard=0.7856, emission=0.9104, quality=0.8480)
  5 -> TssFlnk  (jaccard=0.0001, emission=0.7022, quality=0.3512)
  6 -> ReprPC  (jaccard=0.3111, emission=0.8736, quality=0.5924)
  7 -> Quies  (jaccard=0.9031, emission=0.9931, quality=0.9481)
  8 -> Het  (jaccard=0.7386, emission=0.9523, quality=0.8455)
  9 -> ZNF/Rpts  (jaccard=0.7015, emission=0.9621, quality=0.8318)
  10 -> Tx  (jaccard=0.6757, emission=0.9987, quality=0.8372)
  1

Finished monocytes: 15 strategies, functional benchmarks ['tss_exptss', 'tss_refseq', 'tx']


Incorporating emission similarity (marks: ['H3K27AC', 'H3K27ME3', 'H3K36ME3', 'H3K4ME1', 'H3K4ME3', 'H3K9ME3']; work binarized, reference bigwig)
avg_similarity = 0.3082 (jaccard) 0.8952 (emission) 0.6017 (quality) over 15/15 matched states, bp accuracy 0.4224
  1 -> Biv  (jaccard=0.3431, emission=0.9515, quality=0.6473)
  2 -> ReprPC  (jaccard=0.1695, emission=0.6840, quality=0.4268)
  3 -> Quies  (jaccard=0.4879, emission=0.9967, quality=0.7423)
  4 -> Unknown  (jaccard=0.2731, emission=0.9984, quality=0.6357)
  5 -> ReprPCWk  (jaccard=0.0332, emission=0.8615, quality=0.4473)
  6 -> Het  (jaccard=0.3130, emission=0.7387, quality=0.5258)
  7 -> ZNF/Rpts  (jaccard=0.3982, emission=0.9719, quality=0.6851)
  8 -> TxWk  (jaccard=0.3104, emission=0.9705, quality=0.6404)
  9 -> Tx  (jaccard=0.6561, emission=0.9824, quality=0.8193)
  10 -> EnhG  (jaccard=0.5342, emission=0.9787, quality=0.7564)
  11 -> Enh  (jaccard=0.0803, emission=0.8966, quality=0.4884)
  12 -> EnhLo  (jaccard=0.1253, emi

...

Incorporating emission similarity (marks: ['H3K27AC', 'H3K27ME3', 'H3K36ME3', 'H3K4ME1', 'H3K4ME3', 'H3K9ME3']; work binarized, reference bigwig)
avg_similarity = 0.2700 (jaccard) 0.7922 (emission) 0.5311 (quality) over 15/15 matched states, bp accuracy 0.6977
  1 -> ZNF/Rpts  (jaccard=0.5370, emission=0.8822, quality=0.7096)
  2 -> EnhG2  (jaccard=0.0000, emission=0.1460, quality=0.0730)
  3 -> Het  (jaccard=0.5813, emission=0.8860, quality=0.7337)
  4 -> Quies  (jaccard=0.7240, emission=0.9990, quality=0.8615)
  5 -> TxWk  (jaccard=0.0167, emission=0.9983, quality=0.5075)
  6 -> ReprPC  (jaccard=0.4482, emission=0.9038, quality=0.6760)
  7 -> Biv  (jaccard=0.3205, emission=0.7121, quality=0.5163)
  8 -> TssFlnkD  (jaccard=0.0578, emission=0.9370, quality=0.4974)
  9 -> TssFlnk  (jaccard=0.3392, emission=0.9273, quality=0.6333)
  10 -> Tss  (jaccard=0.5060, emission=0.9994, quality=0.7527)
  11 -> TssFlnkU  (jaccard=0.1098, emission=0.9996, quality=0.5547)
  12 -> Enh2  (jaccard=0.121

Finished substantia_nigra: 15 strategies, functional benchmarks ['tss_refseq']


  RNA-seq: top 12000 expressed genes selected
  GTF: 12000 exp bodies, 48836 non-exp bodies


Finished temporal_lobe: 15 strategies, functional benchmarks ['tss_exptss', 'tss_refseq', 'tx']
used == convex_jaccard_em_0.5_0.5 on 12/12 datasets
Evaluated 12 datasets across 15 strategies.
Datasets per metric: {'accuracy': 12, 'balanced_accuracy': 12, 'accuracy_regulatory': 12, 'accuracy_background': 12, 'mean_matched_emission_cos': 12, 'f1_active_atac': 7, 'f1_tss_exptss': 9, 'f1_tss_refseq': 12, 'f1_tx': 9, 'mean_f1': 7}


Incorporating emission similarity (marks: ['H3K27AC', 'H3K27ME3', 'H3K36ME3', 'H3K4ME1', 'H3K4ME3', 'H3K9ME3']; work binarized, reference bigwig)
avg_similarity = 0.5454 (jaccard) 0.8312 (emission) 0.6883 (quality) over 15/15 matched states, bp accuracy 0.7201
  1 -> ReprPC  (jaccard=0.8168, emission=0.8089, quality=0.8128)
  2 -> Quies  (jaccard=0.7238, emission=1.0000, quality=0.8619)
  3 -> Het  (jaccard=0.5395, emission=0.7750, quality=0.6572)
  4 -> ZNF/Rpts  (jaccard=0.6659, emission=0.8841, quality=0.7750)
  5 -> Tx  (jaccard=0.4443, emission=0.9993, quality=0.7218)
  6 -> EnhG2  (jaccard=0.0021, emission=0.2187, quality=0.1104)
  7 -> EnhG1  (jaccard=0.2569, emission=0.9217, quality=0.5893)
  8 -> Biv  (jaccard=0.0000, emission=0.0039, quality=0.0020)
  9 -> TxWk  (jaccard=0.7184, emission=0.9505, quality=0.8345)
  10 -> Enh2  (jaccard=0.7642, emission=0.9913, quality=0.8778)
  11 -> Enh1  (jaccard=0.5920, emission=0.9888, quality=0.7904)
  12 -> TssFlnkU  (jaccard=0.6934, emis

## 2. Paired differences against overlap matching

Strategies share datasets, so they are compared per dataset against **Overlap only**: mean Δ, the number of datasets whose mapping differs from the overlap mapping at all, how many of those are better / worse, and the Wilcoxon signed-rank p over the datasets where the metric changed. A strategy that changes few mappings can only move the means by a few datasets.

In [4]:
DELTA_METRICS = ["balanced_accuracy", "accuracy_regulatory", "mean_matched_emission_cos"] + \
                [f"f1_{name}" for name in FUNCTIONAL]
values = {m: df_results.pivot(index="dataset", columns="strategy", values=m) for m in ["accuracy"] + DELTA_METRICS}
delta_rows = []
for s in strategy_order:
    if s == "overlap":
        continue
    changed = maps[s] != maps["overlap"]
    row = {"strategy": s, "label": strategy_labels[s], "mappings_changed": f"{changed.sum()}/{len(changed)}"}
    for m in ["accuracy"] + DELTA_METRICS:
        d = (values[m][s] - values[m]["overlap"]).dropna()
        nz = d[d.abs() > 1e-12]
        row[f"d_{m}"] = d.mean()
        row[f"{m}_better_worse"] = f"{(nz > 0).sum()}/{(nz < 0).sum()} of {len(d)}"
        row[f"p_{m}"] = wilcoxon(nz).pvalue if len(nz) >= 2 else np.nan
    delta_rows.append(row)
df_delta = pd.DataFrame(delta_rows)
df_delta.round(4).to_csv(f"{OUT}/matching_strategies_vs_overlap.tsv", sep="\t", index=False)

## 3. Plots

- **Sensitivity vs coverage** per functional benchmark: recall of the annotation by the mapped state family against its precision; faint points are datasets, markers mean ± SE.
- **Trade-offs**: balanced accuracy against the mean functional F1 (datasets with all four benchmarks only), and emission similarity against regulatory-state accuracy. Lines connect the weight series of one family, from $\alpha = 1$ to $\alpha = 0.2$; in the summary panels the $\alpha = 0.2$ end joins **Emissions only**.
- **Emission similarity** of the matched pairs, and **regulatory vs background accuracy**, per strategy.

In [5]:
def scatter_strategies(ax, df, x_col, y_col, scale=1.0, show_points=True):
    # Draw every strategy as points + mean ± SE (or the mean alone); return {strategy: (x, y)}.
    sub_df = df.dropna(subset=[x_col, y_col])
    means = {}
    for strat in strategy_order:
        s_data = sub_df[sub_df["strategy"] == strat]
        if s_data.empty:
            continue
        x_vals, y_vals = s_data[x_col] * scale, s_data[y_col] * scale
        means[strat] = (x_vals.mean(), y_vals.mean())
        marker, color = strategy_markers[strat], strategy_colors[strat]
        if show_points:
            ax.scatter(x_vals, y_vals, color=color, marker=marker, alpha=0.3, s=utils.POINT_SIZE, edgecolors="none")
            ax.errorbar(*means[strat], xerr=x_vals.sem() if len(x_vals) > 1 else 0,
                        yerr=y_vals.sem() if len(y_vals) > 1 else 0, fmt=marker, color=color,
                        label=strategy_labels[strat], markersize=7.5, markeredgecolor="white",
                        markeredgewidth=1.2, zorder=5)
        else:
            ax.scatter(*means[strat], color=color, marker=marker, label=strategy_labels[strat],
                       s=90 if strat == "used" else 70, edgecolor="white", linewidth=1.2, zorder=5)
    for group, color in linear_combination_groups:
        pts = [means[s] for s in group if s in means]
        if len(pts) > 1:
            ax.plot(*zip(*pts), color=color, linestyle="--", linewidth=1.5, alpha=0.8, zorder=4)
    if not show_points and "emissions" in means:
        for group, _ in linear_combination_groups:
            if group[-1] in means:
                ax.plot(*zip(means[group[-1]], means["emissions"]), color="#B0B0B0",
                        linestyle="--", linewidth=1.2, alpha=0.8, zorder=3)
    return means


def strategy_scatter(df, x_col, y_col, title, xlabel, ylabel, path, scale=1.0, show_points=True):
    fig, ax = plt.subplots(figsize=(7.475, 5))
    scatter_strategies(ax, df, x_col, y_col, scale=scale, show_points=show_points)
    n = df.dropna(subset=[x_col, y_col])["dataset"].nunique()
    ax.set_title(f"{title} (n={n})", **utils.TITLE_STYLE)
    ax.set_xlabel(xlabel, fontsize=utils.AXIS_FONTSIZE)
    ax.set_ylabel(ylabel, fontsize=utils.AXIS_FONTSIZE)
    ax.tick_params(labelsize=utils.TICK_FONTSIZE)
    ax.grid(alpha=0.3)
    ax.legend(title="Strategy", fontsize=utils.LEGEND_FONTSIZE, title_fontsize=utils.LEGEND_TITLE_FONTSIZE,
              bbox_to_anchor=(1.01, 1), loc="upper left", borderaxespad=0)
    utils.save_fig(fig, path)


VALIDATION_PLOTS = [
    ("active_atac", "Active chromatin vs ATAC-seq", "ATAC-seq peak bp covered (%)", "Active state bp in peaks (%)"),
    ("tss_exptss", "Tss states vs expressed TSS", "Expressed TSS bp covered (%)", "Tss state bp at expressed TSS (%)"),
    ("tss_refseq", "Tss states vs RefSeq TSS", "RefSeq TSS bp covered (%)", "Tss state bp at RefSeq TSS (%)"),
    ("tx", "Tx states vs expressed gene bodies", "Expressed gene body bp covered (%)", "Tx state bp in expressed genes (%)"),
]
for name, title, xlabel, ylabel in VALIDATION_PLOTS:
    strategy_scatter(df_results, f"recall_{name}", f"precision_{name}", title, xlabel, ylabel,
                     f"{OUT}/summary_2way_{name}.pdf", scale=100.0)

TRADEOFF_PLOTS = [
    ("balanced_accuracy", "mean_f1", "Balanced accuracy vs functional F1",
     "Balanced accuracy (mean recall over reference states)", "Mean functional F1", "balanced_accuracy_vs_f1"),
    ("mean_matched_emission_cos", "accuracy_regulatory", "Emission similarity vs regulatory accuracy",
     "Mean matched emission similarity", "Regulatory state bp accuracy", "cosine_vs_regulatory"),
]
for x_col, y_col, title, xlabel, ylabel, key in TRADEOFF_PLOTS:
    strategy_scatter(df_results, x_col, y_col, title, xlabel, ylabel, f"{OUT}/matching_tradeoff_{key}.pdf")
    strategy_scatter(df_results, x_col, y_col, f"{title}, means", xlabel, ylabel,
                     f"{OUT}/matching_tradeoff_{key}_summary.pdf", show_points=False)

utils.bar_plot(df_results, x="strategy", y="mean_matched_emission_cos", order=strategy_order,
               palette=strategy_colors, figsize=(14, 5.5), hatch=None,
               title=f"Emission similarity of matched states (n={df_results['dataset'].nunique()})",
               ylabel="Mean matched emission similarity", xticklabels=[strategy_labels[s] for s in strategy_order],
               points=dict(size=utils.STRIP_SIZE), path=f"{OUT}/matching_semantic.pdf")

CATEGORY_LABELS = {"accuracy_regulatory": "Regulatory (promoter, enhancer, bivalent)",
                   "accuracy_background": "Background (Quies, Het, ZNF/Rpts, ReprPC)"}
df_reg_bg = df_results.melt(id_vars=["strategy", "dataset"], value_vars=list(CATEGORY_LABELS),
                            var_name="category", value_name="acc_value")
df_reg_bg["category_label"] = df_reg_bg["category"].map(CATEGORY_LABELS)
utils.bar_plot(df_reg_bg, x="strategy", y="acc_value", order=strategy_order,
               hue="category_label", hue_order=list(CATEGORY_LABELS.values()),
               palette=dict(zip(CATEGORY_LABELS.values(), ["#3366CC", "#999999"])),
               figsize=(16, 5.5), title=f"Regulatory vs background bp accuracy (n={df_results['dataset'].nunique()})",
               ylabel="BP accuracy", xticklabels=[strategy_labels[s] for s in strategy_order],
               hatch=None, legend=True, legend_title="Reference states", points=dict(size=utils.STRIP_SIZE, alpha=0.5),
               path=f"{OUT}/matching_regulatory_vs_background.pdf")

  saved /Users/Oleg.Shpynov/data/2026_segmentations/encode/match/summary_2way_active_atac.pdf
  saved /Users/Oleg.Shpynov/data/2026_segmentations/encode/match/summary_2way_tss_exptss.pdf
  saved /Users/Oleg.Shpynov/data/2026_segmentations/encode/match/summary_2way_tss_refseq.pdf
  saved /Users/Oleg.Shpynov/data/2026_segmentations/encode/match/summary_2way_tx.pdf
  saved /Users/Oleg.Shpynov/data/2026_segmentations/encode/match/matching_tradeoff_balanced_accuracy_vs_f1.pdf
  saved /Users/Oleg.Shpynov/data/2026_segmentations/encode/match/matching_tradeoff_balanced_accuracy_vs_f1_summary.pdf
  saved /Users/Oleg.Shpynov/data/2026_segmentations/encode/match/matching_tradeoff_cosine_vs_regulatory.pdf
  saved /Users/Oleg.Shpynov/data/2026_segmentations/encode/match/matching_tradeoff_cosine_vs_regulatory_summary.pdf
  saved /Users/Oleg.Shpynov/data/2026_segmentations/encode/match/matching_semantic.pdf
  saved /Users/Oleg.Shpynov/data/2026_segmentations/encode/match/matching_regulatory_vs_backgr

## 4. Cross-sample consistency of the transferred labels

Mapping to the reference transfers its labels, so two datasets mapped onto it can be compared label by label. For every pair of datasets (A, B), the **bp accuracy** is the genome share where A and B carry the same label:

- **Via reference**: A and B labelled by the used mapping.
- **Ceiling**: the best direct bijection between A's and B's own states (Hungarian on their overlap). Two bijections through the reference compose to a bijection A → B, so the reference can at best match it; a point on y = x lost nothing in the transfer.
- **Random**: the mean over random bijections A → B, what labels agree by chance.

The summary also gives **Kappa** of the used mapping: the bp accuracy corrected for the agreement two label shares reach by chance. The two scatterplots put bp agreement and Kappa via reference labels against the ARI of A and B, which needs no matching: agreement should rise with it, so each carries its Spearman ρ and p-value, with the random-bijection level in grey and the line y = x. Only datasets that share one reference vocabulary are paired: the Mint references name 5 of their 15 states differently from the ChIP ones, so a ChIP × Mint pair disagrees on those labels whatever the mapping.

In [6]:
N_RANDOM = 100


def parse_mapping(text):
    return dict(item.split("=", 1) for item in text.split(";") if item.split("=", 1)[1])


def accuracy(overlap, map1, map2):
    # Genome share with the same label after renaming both sides; an unmapped state matches nothing.
    total = sum(overlap.values()) or 1
    return sum(bp for (s1, s2), bp in overlap.items()
               if s1 in map1 and s2 in map2 and map1[s1] == map2[s2]) / total



def kappa(overlap, map1, map2):
    # Cohen's kappa of the two relabelled sides: accuracy corrected by the agreement of their label shares.
    total = sum(overlap.values()) or 1
    shares1, shares2 = defaultdict(float), defaultdict(float)
    for (s1, s2), bp in overlap.items():
        shares1[map1.get(s1)] += bp / total
        shares2[map2.get(s2)] += bp / total
    pe = sum(shares1[label] * shares2[label] for label in shares1 if label is not None)
    po = accuracy(overlap, map1, map2)
    return (po - pe) / (1 - pe) if pe < 1 else 1.0


# The largest group of datasets whose references share one state vocabulary.
used_maps = {ds: parse_mapping(maps.loc[ds, "used"]) for ds in maps.index}
vocabularies = defaultdict(list)
for ds in maps.index:
    vocabularies[frozenset(parse_mapping(maps.loc[ds, "overlap"]).values())].append(ds)
cross_datasets = sorted(max(vocabularies.values(), key=len))
print(f"Pairing {len(cross_datasets)} datasets of one reference vocabulary; left out:",
      sorted(set(maps.index) - set(cross_datasets)))

segs = {ds: match.load_bed(work_segmentation_path(workdir, ds, config["datasets"][ds]["cell"]))
        for ds in cross_datasets}
indices = {ds: match.build_index(s) for ds, s in segs.items()}

rng = np.random.default_rng(0)
cross_rows = []
for ds1, ds2 in combinations(cross_datasets, 2):
    overlap = match.pair_overlap(None, segs[ds1], ref_index=indices[ds2])
    s1 = sorted({a for a, _ in overlap}, key=match._natural_sort_key)
    s2 = sorted({b for _, b in overlap}, key=match._natural_sort_key)
    identity = {b: b for b in s2}
    randoms = [dict(zip(s1, rng.permutation(s2))) for _ in range(N_RANDOM)]
    ov = np.array([[overlap.get((a, b), 0) for b in s2] for a in s1], dtype=float)
    cross_rows.append({
        "ds1": ds1, "ds2": ds2, "ari": match.adjusted_rand_index(overlap),
        "used": accuracy(overlap, used_maps[ds1], used_maps[ds2]),
        "used_kappa": kappa(overlap, used_maps[ds1], used_maps[ds2]),
        "ceiling": accuracy(overlap, hungarian(ov, s1, s2), identity),
        "random": float(np.mean([accuracy(overlap, r, identity) for r in randoms])),
        "random_kappa": float(np.mean([kappa(overlap, r, identity) for r in randoms])),
    })
df_cross = pd.DataFrame(cross_rows)
df_cross.to_csv(f"{OUT}/cross_sample_agreement.tsv", sep="\t", index=False)
cross_cols = ["used", "used_kappa", "ceiling", "random", "random_kappa"]
cross_summary = pd.DataFrame({
    "mean": df_cross[cross_cols].mean(),
    "spearman_ari": [spearmanr(df_cross["ari"], df_cross[c]).statistic for c in cross_cols],
    "spearman_p": [spearmanr(df_cross["ari"], df_cross[c]).pvalue for c in cross_cols],
})
cross_summary.round({"mean": 4, "spearman_ari": 4}).to_csv(f"{OUT}/cross_sample_summary.tsv", sep="\t",
                                                          float_format="%.4g")



def ari_scatter(y_col, random_col, ylabel, path):
    # Agreement via reference labels against the matching-free ARI of each dataset pair.
    rho, p = spearmanr(df_cross["ari"], df_cross[y_col])
    fig, ax = plt.subplots(figsize=(5.5, 5))
    ax.scatter(df_cross["ari"], df_cross[random_col], color="#C8C8C8", s=utils.POINT_SIZE, edgecolors="none",
               label="Random bijection")
    ax.scatter(df_cross["ari"], df_cross[y_col], color=utils.MODE_COLORS[utils.FULL_DISPLAY],
               s=2 * utils.POINT_SIZE, edgecolors=utils.POINT_STYLE["edgecolor"],
               linewidths=utils.POINT_STYLE["linewidth"], label="Used mapping")
    ax.plot([0, 1], [0, 1], color="#404040", linestyle="--", linewidth=1, label="y = x")
    ax.set_xlim(0, 1)
    ax.set_ylim(min(0, df_cross[random_col].min() - 0.02), 1)
    ax.text(0.97, 0.03, f"Spearman ρ = {rho:.2f}\np = {p:.2g}", transform=ax.transAxes, va="bottom", ha="right",
            fontsize=utils.ANNOTATION_FONTSIZE + 1, bbox=dict(facecolor="white", edgecolor="#C8C8C8"))
    ax.legend(fontsize=utils.LEGEND_FONTSIZE, loc="upper left")
    ax.set_title(f"{ylabel} vs ARI (n={len(df_cross)} pairs)", **utils.TITLE_STYLE)
    ax.set_xlabel("ARI (no state matching)", fontsize=utils.AXIS_FONTSIZE)
    ax.set_ylabel(f"{ylabel} via reference labels", fontsize=utils.AXIS_FONTSIZE)
    ax.tick_params(labelsize=utils.TICK_FONTSIZE)
    ax.grid(alpha=0.3)
    utils.save_fig(fig, path, bar_width=False)


ari_scatter("used", "random", "bp agreement", f"{OUT}/cross_sample_ari_vs_accuracy.pdf")
ari_scatter("used_kappa", "random_kappa", "Kappa", f"{OUT}/cross_sample_ari_vs_kappa.pdf")


Pairing 10 datasets of one reference vocabulary; left out: ['gm12878_mint', 'monocytes_mint']
  saved /Users/Oleg.Shpynov/data/2026_segmentations/encode/match/cross_sample_ari_vs_accuracy.pdf
  saved /Users/Oleg.Shpynov/data/2026_segmentations/encode/match/cross_sample_ari_vs_kappa.pdf


## 5. Results

In [7]:
show_table(f"{OUT}/matching_strategies_summary.tsv",
           caption="Mean per strategy. Functional F1 columns average over the datasets with that annotation; mean_f1 over those with all four.")
show_table(f"{OUT}/matching_strategies_vs_overlap.tsv",
           caption="Paired differences against overlap matching (better/worse counted over datasets where the metric changed).")
show_table(f"{OUT}/cross_sample_summary.tsv",
           caption="Cross-sample bp accuracy of dataset pairs: used mapping, best direct matching, random bijection, and Kappa of the used mapping; Spearman rho with ARI over pairs and its p-value.")

,strategy,label,accuracy,balanced_accuracy,accuracy_regulatory,accuracy_background,mean_matched_emission_cos,f1_active_atac,f1_tss_exptss,f1_tss_refseq,f1_tx,mean_f1
0,overlap,Overlap only,0.7694,0.6709,0.7266,0.7736,0.8268,0.2867,0.4126,0.2462,0.3882,0.3467
1,jaccard_em_1.0_0.0,Jaccard only,0.7663,0.6663,0.7222,0.7708,0.8271,0.2559,0.4126,0.2462,0.3347,0.3272
2,jaccard_em_0.8_0.2,"Jaccard + Em (0.8, 0.2)",0.7636,0.6634,0.7220,0.7676,0.8428,0.2880,0.4126,0.2462,0.3347,0.3353
3,jaccard_em_0.5_0.5,"Jaccard + Em (0.5, 0.5)",0.7594,0.6626,0.7186,0.7624,0.8497,0.2880,0.4126,0.2697,0.3714,0.3471
4,jaccard_em_0.2_0.8,"Jaccard + Em (0.2, 0.8)",0.7369,0.6562,0.6561,0.7389,0.8625,0.1970,0.4126,0.2697,0.4126,0.3322
5,convex_jaccard_em_1.0_0.0,√Jaccard only,0.7663,0.6675,0.7223,0.7708,0.8279,0.2530,0.4126,0.2462,0.3353,0.3267
6,convex_jaccard_em_0.8_0.2,"√Jaccard + Em (0.8, 0.2)",0.7591,0.6649,0.7171,0.7624,0.8485,0.2880,0.4126,0.2591,0.3714,0.3471
7,convex_jaccard_em_0.5_0.5,"√Jaccard + Em (0.5, 0.5)",0.7406,0.6681,0.7118,0.7389,0.8520,0.2880,0.4126,0.2591,0.3714,0.3471
8,convex_jaccard_em_0.2_0.8,"√Jaccard + Em (0.2, 0.8)",0.7363,0.6587,0.6481,0.7389,0.8628,0.1970,0.4126,0.2591,0.4126,0.3322
9,npmi_em_1.0_0.0,NPMI only,0.7365,0.6633,0.6379,0.7389,0.8525,0.2865,0.4167,0.2630,0.3411,0.3397


,strategy,label,mappings_changed,d_accuracy,accuracy_better_worse,p_accuracy,d_balanced_accuracy,balanced_accuracy_better_worse,p_balanced_accuracy,d_accuracy_regulatory,...,p_f1_active_atac,d_f1_tss_exptss,f1_tss_exptss_better_worse,p_f1_tss_exptss,d_f1_tss_refseq,f1_tss_refseq_better_worse,p_f1_tss_refseq,d_f1_tx,f1_tx_better_worse,p_f1_tx
0,jaccard_em_1.0_0.0,Jaccard only,7/12,-0.0030,0/7 of 12,0.0156,-0.0046,1/6 of 12,0.2969,-0.0043,...,NaN,0.0000,0/0 of 9,NaN,0.0000,0/0 of 12,NaN,-0.0534,0/2 of 9,0.50
1,jaccard_em_0.8_0.2,"Jaccard + Em (0.8, 0.2)",9/12,-0.0058,0/9 of 12,0.0039,-0.0075,1/8 of 12,0.1289,-0.0046,...,NaN,0.0000,0/0 of 9,NaN,0.0000,0/0 of 12,NaN,-0.0534,0/2 of 9,0.50
2,jaccard_em_0.5_0.5,"Jaccard + Em (0.5, 0.5)",10/12,-0.0100,0/10 of 12,0.0020,-0.0083,2/8 of 12,0.1055,-0.0080,...,NaN,0.0000,0/0 of 9,NaN,0.0235,1/0 of 12,NaN,-0.0167,0/1 of 9,NaN
3,jaccard_em_0.2_0.8,"Jaccard + Em (0.2, 0.8)",10/12,-0.0324,0/10 of 12,0.0020,-0.0147,3/7 of 12,0.2324,-0.0705,...,0.3125,0.0000,0/0 of 9,NaN,0.0235,1/0 of 12,NaN,0.0245,2/0 of 9,0.50
4,convex_jaccard_em_1.0_0.0,√Jaccard only,7/12,-0.0030,0/7 of 12,0.0156,-0.0034,1/6 of 12,0.2969,-0.0043,...,NaN,0.0000,0/0 of 9,NaN,0.0000,0/0 of 12,NaN,-0.0528,0/2 of 9,0.50
5,convex_jaccard_em_0.8_0.2,"√Jaccard + Em (0.8, 0.2)",9/12,-0.0103,0/9 of 12,0.0039,-0.0060,1/8 of 12,0.1289,-0.0094,...,NaN,0.0000,0/0 of 9,NaN,0.0129,1/0 of 12,NaN,-0.0167,0/1 of 9,NaN
6,convex_jaccard_em_0.5_0.5,"√Jaccard + Em (0.5, 0.5)",10/12,-0.0287,0/10 of 12,0.0020,-0.0028,3/7 of 12,0.1934,-0.0148,...,NaN,0.0000,0/0 of 9,NaN,0.0129,1/0 of 12,NaN,-0.0167,0/1 of 9,NaN
7,convex_jaccard_em_0.2_0.8,"√Jaccard + Em (0.2, 0.8)",10/12,-0.0330,0/10 of 12,0.0020,-0.0121,4/6 of 12,0.3223,-0.0785,...,0.3125,0.0000,0/0 of 9,NaN,0.0129,1/0 of 12,NaN,0.0245,2/0 of 9,0.50
8,npmi_em_1.0_0.0,NPMI only,11/12,-0.0328,0/11 of 12,0.0010,-0.0076,4/7 of 12,0.2783,-0.0887,...,1.0000,0.0042,1/0 of 9,NaN,0.0169,3/0 of 12,0.25,-0.0471,0/3 of 9,0.25
9,npmi_em_0.8_0.2,"NPMI + Em (0.8, 0.2)",11/12,-0.0328,0/11 of 12,0.0010,-0.0066,4/7 of 12,0.4131,-0.0887,...,1.0000,0.0042,1/0 of 9,NaN,0.0162,2/0 of 12,0.50,-0.0207,1/2 of 9,0.50


,Unnamed: 0,mean,spearman_ari,spearman_p
0,used,0.5890,0.7507,2.869000e-09
1,used_kappa,0.2542,0.8518,1.193000e-13
2,ceiling,0.6250,0.8443,3.189000e-13
3,random,0.0658,-0.0746,6.264000e-01
4,random_kappa,0.0041,0.0532,7.284000e-01


True

## 6. IMR90 state colour legends

Legend-like colour panels for the published IMR90 ENCODE reference states and the IMR90 de-novo 15-state ChromHMM model.

In [21]:
def rgb_to_hex(rgb):
    return "#" + "".join(f"{int(channel):02X}" for channel in rgb.split(","))


def load_bed_state_colors(path, key=match._natural_sort_key):
    state_colors = {}
    with open(path) as fh:
        for line in fh:
            if line.startswith("track"):
                continue
            fields = line.split()
            if len(fields) >= 9 and fields[3] not in state_colors:
                state_colors[fields[3]] = rgb_to_hex(fields[8])
            if len(state_colors) == 15:
                break
    return {state: state_colors[state] for state in sorted(state_colors, key=key)}


def plot_state_color_row(state_colors, path, label_rotation=45, width_per_state=0.42, square_size=520):
    labels = list(state_colors)
    x = np.arange(len(labels))
    square_inches = np.sqrt(square_size) / 72
    fig, ax = plt.subplots(figsize=(width_per_state * len(labels) + 0.5, square_inches * 1.1))
    fig.subplots_adjust(top=1, bottom=0)

    ax.scatter(
        x,
        np.zeros(len(labels)),
        marker="s",
        s=square_size,
        c=[state_colors[label] for label in labels],
        edgecolors="#404040",
        linewidths=0.7,
    )
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=label_rotation, ha="right" if label_rotation else "center", rotation_mode="anchor")
    ax.set_yticks([])
    ax.set_ylim(-0.5, 0.5)
    ax.set_xlim(-0.5, len(labels) - 0.5)
    ax.tick_params(axis="x", labelsize=utils.TICK_FONTSIZE, length=0, pad=2)
    for spine in ax.spines.values():
        spine.set_visible(False)
    utils.save_fig(fig, path, bar_width=False, bbox_inches="tight", pad_inches=0.02)


imr90_ref_bed = os.path.join(workdir, "imr90", f'{config["datasets"]["imr90"]["ref_chromhmm"]}_chromhmm.bed')
IMR90_ENCODE_STATE_COLORS = load_bed_state_colors(imr90_ref_bed, key=summary_plots.STATE_IDX.get)
imr90_dense_bed = work_segmentation_path(workdir, "imr90", config["datasets"]["imr90"]["cell"])
IMR90_CHROMHMM_DENOVO_STATE_COLORS = load_bed_state_colors(imr90_dense_bed)

plot_state_color_row(
    IMR90_ENCODE_STATE_COLORS,
    f"{OUT}/imr90_reference_states.pdf",
)
plot_state_color_row(
    IMR90_CHROMHMM_DENOVO_STATE_COLORS,
    f"{OUT}/imr90_chromhmm_denovo_states.pdf",
    label_rotation=0,
)

  saved /Users/Oleg.Shpynov/data/2026_segmentations/encode/match/imr90_reference_states.pdf
  saved /Users/Oleg.Shpynov/data/2026_segmentations/encode/match/imr90_chromhmm_denovo_states.pdf


/Users/Oleg.Shpynov/work/omni-chromhmm/scripts/analysis/utils.py:1407: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  fig.tight_layout()
/Users/Oleg.Shpynov/work/omni-chromhmm/scripts/analysis/utils.py:1407: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  fig.tight_layout()
